In [1]:
# ============================================================
# SECTION 0 — Install Dependencies
# ============================================================

!pip install scikit-learn networkx sentence-transformers
!pip install transformers accelerate bitsandbytes pandas
!pip install -U spacy
!pip install -U scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

  Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.5 kB)
Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
Using cached confection-1.3.3-py3-none-any.whl (35 kB)
Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.9 MB)
Using cached weasel-1.0.0-py3-none-any.whl (50 kB)
Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.4 MB)
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5
    Uninstalling confection-0.1.5:
      Successfully uninstalled confection-0.1.5
  At

In [2]:
# ============================================================
# SECTION 1 — PATH SETUP
# ============================================================

import sys
import os
import pickle
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

sys.path.append(SRC_PATH)

print("SRC path added:", SRC_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SRC path added: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/src


In [3]:
# ============================================================
# SECTION 2 — IMPORT MODULES
# ============================================================

from router import hybrid_router, extract_query_entities
from local_traversal import run_local_query
from global_retrieval import run_global_query
from viz import visualize_local_traversal, visualize_section_scores, visualize_global_communities, visualize_hybrid_flow, print_top_paths

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/allenai-specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SPECTER loaded


In [4]:
# ============================================================
# SECTION 3 — LOAD GRAPH
# ============================================================

GRAPH_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "aria_lite_graph_v2_1_communities_v1.pkl"
)

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Graph loaded
Nodes: 2105
Edges: 22506


In [5]:
# ============================================================
# SECTION 4 — LOAD LLM (PHI-3 MINI)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("LLM loaded")

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

LLM loaded


In [6]:
# ============================================================
# SECTION 5 — MAIN PIPELINE FUNCTION
# ============================================================

def run_pipeline(query, top_k=3):

    print("\n==============================")
    print("QUERY")
    print("==============================")
    print(query)

    # --------------------------------------------------------
    # STEP 1 — ROUTING
    # --------------------------------------------------------

    route = hybrid_router(query)

    print("\nROUTE:", route)

    # --------------------------------------------------------
    # STEP 2 — CONTEXT BUILDING
    # --------------------------------------------------------

    context = ""

    if route == "LOCAL":

        results = run_local_query(G, query)

        top_sections = results[:top_k]

        for r in top_sections:
            context += "\nSECTION:\n"
            context += r["text"] + "\n"

    else:

        retrieved, context_pack = run_global_query(
            G,
            query,
            top_k=top_k
        )

        for community in context_pack:

            context += "\nCOMMUNITY SUMMARY:\n"
            context += community["summary"] + "\n"

            for sec in community["sections"]:
                context += "\nSECTION:\n"
                context += sec["text"] + "\n"

    print("\n==============================")
    print("CHECK CONTENT")
    print("==============================")
    print("QUERY:")
    print(query)
    print("CONTEXT:")
    print(context)
    print("==============================\n\n\n\n")

    # --------------------------------------------------------
    # STEP 3 — LLM ANSWER GENERATION
    # --------------------------------------------------------

    prompt = f"""
              You are a biomedical AI assistant.

              Use ONLY the context below to answer the question.

              Context:
              {context}

              Question:
              {query}

              Provide a clear, accurate, and concise answer in bullet points. Add your answer in the following format:

              ANSWER: <answer>
              """
    output = llm(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    answer = output[0]["generated_text"][len(prompt):].strip()

    # --------------------------------------------------------
    # STEP 4 — OUTPUT
    # --------------------------------------------------------

    print("\n==============================")
    print("FINAL ANSWER")
    print("==============================")
    print(answer)

    return {
        "route": route,
        "answer": answer,
        "context": context
    }

In [7]:
# ============================================================
# SECTION 5 — EXPLAINABILITY WRAPPER
# ============================================================

def explain_query(query):

    route = hybrid_router(query)

    print("\n==============================")
    print("EXPLAINABILITY VIEW")
    print("==============================")

    print("Route:", route)

    # ========================================================
    # LOCAL PIPELINE
    # ========================================================
    if route == "LOCAL":

        results = run_local_query(G, query)

        print("\nTop Retrieved Sections:")
        for r in results[:3]:
            print("\nSECTION:", r["section_id"])
            print("SCORE:", r["score"])
            print("TEXT:", r["text"][:300])

        # -----------------------------
        # VISUALIZATION (LOCAL)
        # -----------------------------
        print("\n[VISUALIZATION] Local Traversal Paths")

        visualize_local_traversal(results[0]["paths"])   # key fix: use paths
        visualize_section_scores(results)

        print_top_paths(results[0]["paths"], top_n=10)

        return results

    # ========================================================
    # GLOBAL PIPELINE
    # ========================================================
    else:

        retrieved, context_pack = run_global_query(G, query)

        print("\nTop Retrieved Communities:")

        for c in retrieved:
            print("\nCOMMUNITY:", c["community_id"])
            print("SCORE:", c["score"])
            print("SUMMARY:", c["summary"][:300])

        # -----------------------------
        # VISUALIZATION (GLOBAL)
        # -----------------------------
        print("\n[VISUALIZATION] Global Community Retrieval")

        visualize_global_communities(retrieved)

        # hybrid view needs both signals
        visualize_hybrid_flow(
            local_results=[],   # empty since not used
            global_results=retrieved
        )

        return retrieved, context_pack

In [8]:
# ============================================================
# SECTION 6 — TEST RUN
# ============================================================

query = "How does trastuzumab target HER2 in breast cancer?"

output = run_pipeline(query)

# results = explain_query(query)


QUERY
How does trastuzumab target HER2 in breast cancer?

ROUTE: LOCAL


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHECK CONTENT
QUERY:
How does trastuzumab target HER2 in breast cancer?
CONTEXT:

SECTION:
p95HER2 is commonly expressed in human epidermal growth factor receptor 2 (HER2)-positive breast cancer (BC) and is generated primarily from shedding of the extracellular domain of HER2. p95HER2 is oncogenic, driving aggressive tumor growth, and conferring drug resistance. Targeting p95HER2 is challenging. Clinically approved HER2 inhibitors either cannot bind to p95HER2 or show limited inhibitory effects. We used cell lines and orthotopic tumor models (cell line xenografts and patient derived xenografts) to investigate the effects of HER2 inhibitors on p95HER2 and other key signaling proteins in HER2-positive BC, and to compare the therapeutic activities of different HER2 inhibitors. The HER2 inhibitors represent different mechanisms of actions, including trastuzumab, pertuzumab, tucatinib, and lapatinib, all of which are clinically approved, as well as PEPDG278D, a recombinant human protein wh

In [9]:
# ============================================================
# SECTION 7 — TEST RUN
# ============================================================

query = "Compare supervised and unsupervised learning for cancer subtyping"

output = run_pipeline(query)

# results = explain_query(query)


QUERY
Compare supervised and unsupervised learning for cancer subtyping

ROUTE: GLOBAL
Communities found: 11


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Communities used: 3

CHECK CONTENT
QUERY:
Compare supervised and unsupervised learning for cancer subtyping
CONTEXT:

COMMUNITY SUMMARY:
Scientific Summary:
              The primary focus of this biomedical research is the application of machine learning (ML) techniques, specifically random forest and XGBoost, to enhance the prognostic modeling of invasive ductal carcinoma (IDC), a predominant form of breast cancer. The study integrates multi-omics data, including transcriptomics and single-cell RNA-seq, to identify key predictors of mortality and recurrence in IDC patients. By leveraging the predictive power of ML models, the research aims to improve individualized treatment strategies, such as surgical and therapeutic decision-making, for IDC patients. The study also explores the potential of 1,2,4-oxadiazole derivatives as dual inhibitors of H

SECTION:
Six independent predictors were identified: visceral adipose tissue density, skeletal muscle density, intramuscular adipose tissue